# AI-Generated News Text Detection: Generalisation to Unseen LLMs

**Research question (proposed):** Does a supervised detector trained to distinguish human-written from AI-generated news text generalise to an LLM it has never seen during training?

Dataset: *A Comprehensive Dataset for Human vs. AI Generated Text Detection* (Roy et al., 2025, arXiv:2510.22874), Hugging Face: `gsingh1-py/train`, CC BY 4.0.

## Notebook structure
1. **Part 1: Setup, data loading and data inspection** *(this version)*
2. Part 2: Preprocessing and prompt-grouped split *(to be added)*
3. Part 3: Baselines: length-only, TF-IDF + Logistic Regression *(to be added)*
4. Part 4: DistilBERT fine-tuning *(to be added)*
5. Part 5: Experiments: seen-generator vs leave-one-generator-out *(to be added)*
6. Part 6: Error analysis *(to be added)*

---

# Part 1: Setup, data loading and data inspection

Goal: understand the raw data *before* any modelling decision. We check:
1. the structure of the dataset (columns, missing values);
2. how the wide table (1 row = 1 prompt, 7 texts) maps to our learning samples (1 row = 1 text);
3. text lengths per source (risk: length shortcut);
4. duplicates and very short texts (data quality);
5. formatting artefacts (e.g. titles, markdown) that could let a detector "cheat";
6. what 7 texts of the same prompt look like (why we must split by prompt).

## 1. Environment setup

In [ ]:
# Colab already has pandas/numpy; `datasets` is the Hugging Face library used to download the data.
!pip install -q datasets

import random
import re
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42                      # fixed seed so every sampling step is reproducible
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## 2. Load the raw dataset

The dataset has a single `train` split (the authors did not provide train/val/test splits,
so we will create our own later, grouped by prompt).

In [ ]:
raw = load_dataset("gsingh1-py/train", split="train")
df_wide = raw.to_pandas()

print("Shape (rows, columns):", df_wide.shape)
print("\nColumns:")
for c in df_wide.columns:
    print("  ", repr(c))

### 2.1 Rename columns to short, consistent names

The original column names are long (e.g. `accounts/yi-01-ai/models/yi-large`).
We map them to short source names. `human` = New York Times article; the other 6 are LLM generators.

In [ ]:
RENAME = {
    "prompt": "prompt",
    "Human_story": "human",
    "gemma-2-9b": "gemma",
    "mistral-7B": "mistral",
    "qwen-2-72B": "qwen",
    "llama-8B": "llama",
    "accounts/yi-01-ai/models/yi-large": "yi",
    "GPT_4-o": "gpt4o",
}

# Safety check: stop immediately if the column names differ from what we expect,
# instead of silently producing wrong data.
missing = set(RENAME) - set(df_wide.columns)
assert not missing, f"Unexpected column names, missing: {missing}"

df_wide = df_wide.rename(columns=RENAME)[list(RENAME.values())]

# prompt_id identifies the group of 7 texts written from the same NYT abstract.
# It is the key for the leakage-free (grouped) split later.
df_wide.insert(0, "prompt_id", np.arange(len(df_wide)))

SOURCES = ["human", "gemma", "mistral", "qwen", "llama", "yi", "gpt4o"]
GENERATORS = SOURCES[1:]
df_wide.head(3)

### 2.2 Missing and empty values per column

In [ ]:
def is_missing(s: pd.Series) -> pd.Series:
    # A text counts as missing if it is null OR only whitespace.
    return s.isna() | (s.astype(str).str.strip() == "")

missing_table = pd.DataFrame({
    "n_missing": {c: int(is_missing(df_wide[c]).sum()) for c in ["prompt"] + SOURCES},
})
missing_table["n_present"] = len(df_wide) - missing_table["n_missing"]
missing_table

## 3. Reshape: wide table → one row per text

**Why:** the learning sample for our detector is *one text*, not one prompt.

- Input `x` = the text
- Target `y` = 0 if written by a human, 1 if generated by an LLM
- `source` and `prompt_id` are **metadata only** (not model inputs): `prompt_id` is used for the grouped split, `source` for the leave-one-generator-out experiment.

In [ ]:
df = df_wide.melt(
    id_vars=["prompt_id", "prompt"],
    value_vars=SOURCES,
    var_name="source",
    value_name="text",
)

# Drop missing / empty texts
df = df[~is_missing(df["text"])].copy()
df["text"] = df["text"].astype(str)

# Label: 0 = human, 1 = AI
df["label"] = (df["source"] != "human").astype(int)

df = df.reset_index(drop=True)
print("Total texts:", len(df))
print("\nTexts per source:")
print(df["source"].value_counts().reindex(SOURCES))
print("\nClass balance (label):")
print(df["label"].value_counts().rename({0: "human (0)", 1: "AI (1)"}))
print(f"\nShare of AI texts: {df['label'].mean():.3f}  "
      "-> an 'always predict AI' classifier would reach this accuracy.")

## 4. Text length per source

**Why this matters:** if human and AI texts differ strongly in length, a detector can learn
"long = human, short = AI" instead of learning writing style. This is a *shortcut*:
it lowers the training loss but fails the real objective.

In [ ]:
df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()

length_stats = (
    df.groupby("source")["n_words"]
      .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
      .reindex(SOURCES)
      .round(0)
)
length_stats

### 4.1 How many texts would be affected by truncation?

A transformer like DistilBERT accepts at most 512 tokens. As a rough guide,
1 English word ≈ 1.3 tokens, so 512 tokens ≈ 390 words and 256 tokens ≈ 195 words.
(This is an approximation; we will measure real token counts with the tokenizer later.)

In [ ]:
for limit in [150, 195, 390]:
    share = (df["n_words"] > limit).groupby(df["source"]).mean().reindex(SOURCES)
    print(f"Share of texts longer than {limit} words:")
    print(share.round(3).to_string(), "\n")

## 5. Data quality: very short texts and duplicates

In [ ]:
# Very short texts may be scraping errors or refusals, not real articles.
SHORT_WORDS = 50
short = df[df["n_words"] < SHORT_WORDS]
print(f"Texts with fewer than {SHORT_WORDS} words, per source:")
print(short["source"].value_counts().reindex(SOURCES, fill_value=0), "\n")

# Exact duplicates (after trimming whitespace)
norm = df["text"].str.strip()
dup_mask = norm.duplicated(keep=False)
print("Texts that have an exact duplicate somewhere:", int(dup_mask.sum()))
print(df[dup_mask]["source"].value_counts().reindex(SOURCES, fill_value=0))

In [ ]:
# Look at a few of the shortest texts to understand what they are
df.sort_values("n_words").groupby("source").head(2)[["source", "n_words", "text"]].head(14)

## 6. Formatting artefacts

LLM outputs often contain formatting that human NYT articles do not (e.g. a `Title:` line,
markdown `**bold**`, `#` headings). If such patterns exist, a detector can use them as a
shortcut. We measure how often each pattern appears per source.

In [ ]:
PATTERNS = {
    "starts_with_title":  r"^\s*(?:\*\*)?\s*title\s*:",   # 'Title:' or '**Title:' at the start
    "markdown_bold":      r"\*\*[^*]+\*\*",               # **bold**
    "markdown_heading":   r"(?:^|\n)\s*#{1,6}\s",            # # heading
    "bullet_list":        r"(?:^|\n)\s*[-*•]\s+\w",          # - item
    "double_newline":     r"\n\s*\n",                       # paragraph breaks
    "any_newline":        r"\n",
}

artefacts = pd.DataFrame({
    name: df["text"].str.contains(pat, flags=re.IGNORECASE, regex=True)
                    .groupby(df["source"]).mean()
    for name, pat in PATTERNS.items()
}).reindex(SOURCES).round(3)
artefacts

In [ ]:
# Most common first 5 words per source: reveals templated openings
def first_words(t, k=5):
    return " ".join(t.split()[:k])

for src in SOURCES:
    top = df.loc[df["source"] == src, "text"].map(first_words).value_counts().head(5)
    print(f"--- {src} ---")
    print(top.to_string(), "\n")

## 7. The 7 texts of one prompt

All 7 texts are about the **same** NYT abstract. This shows why we must split by
`prompt_id`: if some of these texts were in training and others in test, the detector
could recognise the *content* instead of the *writing style* (data leakage).

In [ ]:
example_id = int(df_wide["prompt_id"].sample(1, random_state=SEED).iloc[0])
print("PROMPT (NYT abstract):")
print(df_wide.loc[example_id, "prompt"], "\n")

for src in SOURCES:
    row = df[(df["prompt_id"] == example_id) & (df["source"] == src)]
    if row.empty:
        print(f"===== {src}: MISSING =====\n")
        continue
    t = row["text"].iloc[0]
    print(f"===== {src}  ({len(t.split())} words) =====")
    print(t[:600].replace("\n", " \u23ce "), "...\n")   # \u23ce marks a newline

## 8. Summary to send back

Please run all cells and send back the outputs of:
- Section 2 (shape, columns) and 2.2 (missing values)
- Section 3 (counts, class balance)
- Section 4 and 4.1 (length tables)
- Section 5 (short texts, duplicates, and the shortest-texts table)
- Section 6 (artefact table and first-words lists)
- Section 7 (one example)

These outputs decide the next design choices: truncation length, cleaning rules,
and whether artefact removal is needed before training.